In [ ]:
from pathlib import Path
import importlib.util
import textwrap

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.cluster.hierarchy as sch
import seaborn as sns
import torch
import torch.nn.functional as F
from scipy.spatial.distance import pdist, squareform
from sklearn.manifold import TSNE
from matplotlib.ticker import MultipleLocator
from tokenizers import Tokenizer
from tqdm import tqdm
from transformers import AutoConfig, AutoModelForCausalLM, PreTrainedTokenizerFast

In [ ]:
SINGLE_TEMPLATE = "RNA, {}, ppap2d, phosphatidic acid phosphatase type 2D<seq><gene>TCGAACATGCTGTGACCCCCAGGCTGATCGTTAAACTTCGATTTTGCTGTACACGGGCTCAATTCAACGCGTCAATTCAATGCGGGCACTTCCAAGGATACAACCGAATACCGGTGCAATAAACGTCGTTACCACATGTTATTGTGTACACAAGAGCACTTCCTCCGGCCGTGTTTTACTTTGAAGGCGTAAAACTAACAGTGGGGAGGTCATTGTGCGCGCTTTACAGATGGATGTCTTGTGGGAGAGCAGAGCCAGAACACGCTCCCAGTAAAAGTTTAGGAGATCACAAAAGTTTTAGGTCGTCAGGAGTTTTAATTAACATCCACATCGTTTTACGAAGGTTTCATGGACTTTTACTTTCAACTTTTACTTCAGAAAAACAACTGTGAGTCACCCGCGCTAACAAGCGGCTTTGGACCAAGTTCACGCTGTCAAACTCGTTGACATGGGTTTGCTGCATGGGCGACAGCGAGACTAAAAACTTAGGACTTTTTACACAATCTACCCGCCGCGCACATCTCTCTGAACGCGCTCGTAGCCTATAGCCGGCGGTTTGACACACGGAC<CDS>ATGCAGAAGTTTAACTCCACCGGCGGCCACAGCGACACGCTGCCCCGGGACGCAGACCTCCAGCTGCGGCTGGCTGACTGTGGAGCCTCGGGAGAGAGCAAGGAGAACGGAGCTGGAAAACACTTTCTGAGCCAGCAGCCGGAGCAGACACCTCTGTGCACCAAGAGGCGAGTGCTGGTCGGCCTGGATGTCATCTGCCTCTGTGTTGCCTCTATCCCATTCTTCGCATGTGAGCTGAAAGCAGTGAAAGCATACAGACGTGGGTTCTTCTGCGGAGACACGAGTATCACCTATCCATACATAGAAAGAGAGGCCATTCCTGACAGTCTGCTGATTGCTGGCGGCATTGCTATCACAGGTCTTACAATAGCGCTGGGTGAATGTTTTAGAGTCAGATTCAGAGGTGTGCACTCACGGGCCTTTGTTCGAAACCGATATGTGTCATGTCTATATAAAGAGCTGGGGAGCTTCCTGTTTGGCTGTTGCATTGGTCAGTCCCTCACTAACATGGCCAAGCTGAGTGTGGGAAGACTGCGCCCAAACTTTCTGTCAGTGTGTAATATCACCTACTCGTCCATAAACTGCACACCGGGTAGTTACATCGCTGTGGCCCCATGTCGGCAGAACAACACCAAGCTGGTGGAGGAAGCCAGGAAGTCCTTCTTTTCGGGCCATGCGTCATTTGCCATGTACACCATGATGTATCTAGCATTCTATTTACAAGCTCGGTTGTCATGGCGAGGAGCCCGACTGTTGCGGCCGCTGATTCAGTTCCTGTTGGTGATGATCGCCATCTACACCGGTCTGAGCCGCATCTCCGACTACAGACACCACCCCACCGACGTGCTCACCGGCTTCATCCAGGGGGGTCTCACCGCATACTGGGTGGCGTTCTACATCTCCTCCATGTTTAAACCTTGTGCTCGTCCAGACCTGTCACCCACCTGCATGTCTTTGGAGAGCCCCCTGTCCAGCCAACAAACTGTCTGTTAG</CDS>CACCTACTTAACAATCGTCCAAGCACAGTTAAAATTAAACAGGTAATAACTGGCAATTACTAATGGAAACACAAAACATCTGTCCATCAATTTAATTTCACAGTAATATGGGTAATCTTAATCGCAAATGGGTTTGGCATTAATCCCTTGTCATGGGTACTATGATTACTGGTTGTTAATTCCATTCGACTACT</gene></seq><eos>"

seq1 = "RNA, {}, SCEL, sciellin isoform X4<seq><gene>AGGTAGAGGTTGAGACTCCACTGAATAAACTCTAGGTTCCCATTTCTTTCAGCCAGATCCTCCCAGGGAATCACTACAGGCTGGTTAGCCAAAAAGTCCTGATTTTCTGCTCAATAGAGGTCCTTACTGGAAGGCAGC<CDS>ATGTCCAATGTTACCTTGAGAAAAATGTCTCCCACAGGAAATGAGATGAAGAGCACCACTCAGGGAACCACACGGAAGCAGCAGGATTTTCACGAGGTGAACAAAAGAAGAACTTTCTTACAGGATAACAGTTGGATAAAGAAACGCCCTGAAGAAGAAAAAGATGAAAATTACGGTAGGGTGGTGCTCAACCGACATAATTCCCATGATGCATTGGACAGGAAAGTAAATGAGAGAGATGTGCCAAAAGCTACAATTAGTCGGTACAGTTCTGATGACACTTTGGACAGGATCTCAGACAGAAATGATGCTGCTAAAACATATAAGGCCAATACCTTGGATAACCAACTAACCAATAGGAGCATGTCCATGTTTAGATCACTGGAAGTAACAAAGTTGCAACCTGGCGGTTCATTGAATGCCAACACCTCCAACACCATAGCATCCACTTCTGCTACTACTCCTGTAAAGAAGAAGAGGCAGTCCTGGTTTCCACCGCCCCCTCCAGGTTACAATGCCTCTTCGAGCACAGGAACCAGGAGACGGGAACCAGGTGTTCACCCTCCAATACCTCCAAAGCCCAGTTCTCCTGTTTCTTCTCCTAACCAGCTGAGACAGGATAATAGGCAGATACATCCACCTAAACCAGGTGTATATACAGAAACCAACAGATCTGCTGAAAGAAATATAAGGAGTCAGGATCTTGATAACATCGTCAAAGTGGCCACTTCACTTCAGAGAAGTGACAAAGGTGAAGAATTGGATAATCTCATCAAAATGAACAAAAGCTTGAATAGGAATCAAGGTCTTGATAGTCTCTTCAGAGCAAATCCAAAGGTAGAAGAAAGAGAGAAAAGAGCCAAAAGCCTTGAAAGTCTCATCTATATGAGTACCCGGACAGATAAAGATGGCAAAGGAATCCAAAGCCTTGGAAGTCCGATTAAAGTTAATCAAAGGACTGACAAAAATGAGAAAGGACAAAATCTCGAATCTGTTGCTAAAGTGAATGCCAGGATGAATAAAACGAGCAGAAGTGAAGACCTTGATAATGCTACTGAAGTAAATCCCAAAGGACATGAAAATACCACTGGAAAAAAAGACCTTGATGGGCTTATTAAAGTGGATCCTGAAACAAATAAAAATATTACGAGGGGCCAGAGCCTTGATAATCTCATCAAAGTGACCCCTGAAGTAAAGAGAAGTAACCAAGGTTCCAAAGACCTTAATAACTTCATCAAAGTGTATCCAGGAACAGAAAAAAGTACTGAAGGGGGCCAAAGTCTCGACAGCCTCATTAAAGTGACTCCTGAAAGAAACAGAACTAACCAAGGGAACCAAGACTTGGAAAATCTTATCAAAGTGATCCCTTCAGCAAACAAAAGCAGTGAACAAGGTCTTGATGAACATATTAATGTCAGCCCCAAAGCTGTCAAAAACACTGATGGAAAACAAGATCTTGATAAACTCATCAAGGTGAATCCTGAAATTTTCACAAACAACCAAAGAAACCAAGATCTTGCTAACCTCATCAAAGTAAATCCTGCAGTAATCAGAAACAATCAGAGCCAAGACTTGGACAATCTTATTAAAGTGAAACCTTCAGCTCTTAGAAACACTAATCGAGACCAGAACCTGGAAAATTTAATTGAAGTAAATTCTCATGTGTCTGAAAACAAGAATGGAAGCTCTAACACTGGAGCCAAGCAGGCAGGACCACAGGATACTGTTGTGTACACAAGGACATATGTGGAGAATAGTAAATCACCCAAGGATGGATATCAGGAGAATATCTCTGGAAAATACATACAAACTGTTTATTCAACTTCTGATAGGTCTGTCATTGAAAGAGATATGTGCACTTACTGCCGAAAACCCTTGGGTGTAGAAACTAAAATGATTTTAGATGAATTACAAATTTGCTGCCATTCTACTTGCTTTAAGTGTGAAATATGCAAGCAGCCTTTGGAAAATCTGCAAGCGGGTGATAGTATTTGGATTTATAGACAGACAATACACTGTGAACCTTGCTACTCTAAAATTATGGCAAAGTGGATTCCATAA</CDS>CTCTGGCACAAGGAAATCAAGATGAAAAGCACTCATTAAGGAATTAAAGTTACAAGTTTTATCTTAATAATATGTAATCTAGAAAAGCTTTCACATTGAAGATCAACTCTTGTACAAAATTAACAATTCTGTTATTGCATAAGTAATCTAATTGTCTTCAATAAGGTCACACACATAAAAAGAGCCATCTGGTCTCTGGCTAGAGTTAGCAATAAAAAGTTCAAATGGTTCCAGATTCCAGTGTCAAAGGAGTGATGCATTACACTCCAGCCAGGTCCATCCCTGCTCCGTATGTTGGCTGTGAGTGGTGGTTTCCATTTAAACCAAGTTTCTCATTTCTTCACCTTTTTTTCTCTAAGAATTTGGATTCGTAGACATTGACATCCCGAAGAACTGTCAAGGAAGCAAGATATGCTTTCTTCATCTGCAAAAGAAATACTAACAACAATTTTCTTATACAGTTTGGCAGAAAGATGTTAACATAAAAAGTTTATATACCTCAAAAATCACTAAACTTTCCAGATCTCTGTCCTATTATTTGTAACACAAGGGGCATTGGATAAAATGATTTCTAGGGTTCCTTTTGCTTCCCAAATTCTCTGATTCTAAAGCAGTTTTTAGAATCATTAGCTCTTTGGAAACATATATGCATACATGTTTGTTAAGCCTATTGAACTAGGTAGGACATATAAACAATTTAATTTTAGTGTCATTGTTTAATCACAGACTTAGTGTTTGAAAACTGTGTTTTAAAAACAGAAACAGATTGATGGGTAACAGGTAAAATATGACATGTATAGCTTACATGTTATTATTTGTTAAATTTTCTTTGTATACATTTCAAAATCTGGGTATACTTATAATCCATTAGAAGTAATGGTTATGGACTAAAAAGATATGTTCTTTAGTATGTTATATATACTCATATTACATAGCAGTATGTTTACAAAAGGCTTATAAAAATAAAATGAACTATCAGTTACATAGAA</gene></seq><eos>>"
seq2= "RNA, {}, Vmn1r22, vomeronasal 1 receptor 22<seq><gene><CDS>ATGTCTTCAATCAAGAATGTTCTTTATTTCCAATCTGGACTTGGAGTTCTAGCCAATATGTTTCTACTGTTTTTTTATATTTTCATAATCCTAGGTCACAGACCTAAGCCCATGGACCTGATCTTATGTCAATTTTCCTTTGTTCACATACTGATGTTCCTCACTGCAGGGGATATTTGGCTTTCAGACTTATTTGAGTCACTGAACATTGAGAATGACTTCATATGTAAGGCAACTTTTTACACTGGTAGGGTGATGAGAGGACTCTCTATGTGCATCACATGCCTCCTGGGTGTGTTCCAGGCTGTCACTATCAGTCCCAGTACCTCACTGTTGGCAAAATTTAAGCATAAACTAAAAAAATACATCATCTATGCTGTTTTCTATATTTGGTTTCTTAATTTCTCGATCAATAGTCACATGATAGTCTATGTTGGTGGTTTTAACAATAGGAGTGAGACCAACCAGATGAGAATCACTAAAACCTGTTCACACTTTCCCATGAACAACATCATCAAGGAATTGATTTTAACAGTGATAACTTTAAAAGATGTGTTTCTTGTAGGAGTCATGCTGACCACAAGTATATACATGGTGATTATCTTGTATAGACATCAAAGAAAATGCAAGCATCTTCATAGCATCAGGCACCTGAGAACATCCCCTGAGAAAAAGGCCACACAGACCATCCTGCTGCTGCTGGTTTTCTTTGTGGTCATGTACTGGGTGGACCTCATCATCTCATCCACCTCAGTCCTGTTGTGGAGGTATGACCCAATCATCCTGACTGTTCAGAAGTTTGTGGTGAATGTCTATCCCACAATAACTCCTTTGGTACAAATCAGTTCTGATAAGAGAATAATCAATGTGCTGAAAAACTTGTGGCTAAAATGCCACCAGACTGTTTAA</CDS></gene></seq><eos>"
seq3 = "RNA, {}, gcfc2, intron Large complex component GCFC2 isoform X2<seq><gene>AGGAAGTACAAAATACGTCATTTTAAAGCGTTGATTTCACTCAGCATTGGATTTATA<CDS>ATGTTTAATAAGAAGCCGCGCCGAAATTTCCGGCAAAGGAAAAATGAATCCAGTGATGAAGACGACAATGGAAAATTAGCTGCTGGTGAAGGTGGAAAAGATGCGGATTTAAAGGCAGATTTACCCTCAAAAGCAGGGATTTCAAGCAGTTCACGAGAAGCTCGAGAGTGTGACTCGAGTGAGGCAGAAGAGGAGGCTTCAGTTTCAGCACTCCAGACCAAAGACAGAAAACTGCACACAGACAGCAAAACACTCAGCTTCTCTGATGATAAAGACTCTACAGAGGGAGGTTTTCAAGTGAAACGTTCAGTCAACAGAGAAGTTTTGTTCCAAGCGCGGAAAAAAGAGGATTCTCCTGCACCTGCAACCAGTTCTCAAGGAAAGAAGAGTAAGGATGTCAGTCTCTCTGACAGTGATTCTGATGCATCAGCTGAAGACATTCCTCCTGACAGTGATGAAGAAACCTGTTCTACCATGTCAAGTTCCTCCGCTTCTTCACTACCACAACGAGTCATCATTCCTGATGCCAGAAAGATTCATGCTGCCAAAGAGAAAAGGAGACAAGCCAGAGCCCAGCAAGATTACATCTCTCTGGATTCACCTCGAACTCCTGGAAGCTTGAAGGAGGACGAGCTGAGCGATAAGGAACAAGACAGTGACAATGATCTGGATGATCATGAGCGCAGAATTGAGTTTGCACCCAAACCCAAAACTCTGAGAGAGAGAATGGCTGAGAAGATGGGGAGTGATAGTGAGGAGAGCTTTTCTGACAGCCAGGAGGAGGAAGAGCAGCAGATGTGGGAGGAACAACAGATTGGAAAAGGAGTCAAAAGGCACCAAGAATGGCATGCTCCGAAAGCAGCCCGACAGAAGAGAGTGGAAATCCCAGAAACGCTTCCAGTAGTTAGTATTGGTGTCATTAAGAAGAGAATCACTGGAAAATTGGATTCGCTGAGGGAGGTACACAGGGCACGGGAGGCGCAGCTAAGGAGGATGCAGCTGGATGTGGAAATGGCCAAAACCTCTTTAGAGGGCCTTGAAAACAACAATGCAGATGAGCAGCTCCGATTTTACCGCACCATGAACGTTTTCTCACAGAACCTGCTGGAATGCCTATCTGAGAAGATGGCTTTAATAAACTCAGTTGAGCTGGACATGCACGGTCTCTATATTGACCAGGCGGAGGCACTCTTGAGTCAGAGAAGAGAAGCATTACAAGAGGAGTCCTCTCGGATTCAGAAGTTAACTTATAACACTGATTCACACAGCAATGGTGACACAGCCGGAGACTGCAGCCCTACTCAGCAGAGCTTGTCTGAAGATGACATTGGATGTGTGCCGTGTGACTGGGAACCAACGGTCGAGCAGAAGGAGGAGATTGAAAGTAAAAGAGCTGCTCTGCTGAAAAAGGCACAGGAAGTGTTTGCTGATGTTCAAAATGAGTTCTGGGATGTAAAGAAAATCCTCTCCAGATTTGATGAGTGGCGCGTCTCCTTTAAAGACTCTTACAATAATGCCTACATTGGGCTGTGTCTCCCAAAACTCCTGGCACCCTTGATCCGACACCAGCTCATTGGCTGGAACCCTCTACAGGCAGAAAGTGAGGACTTTGAGGCATTGCCATGGTATTGTGCTGTTGAGAGGTTCTGTCATGGTCAGGGCTACGAGGAATCAGAGAACATGGACAAGACCACACTGCCCACCATAATTGAGAAGACCATCCTCTCTAAAGTTCAAGGTTTTGTAGAGCTGGTCTGGGACCCTCTTTCTGCCCAGCAGACTCGAACCCTGACAACACTATGCAGAAGAATACAGCATGACTATTCTGTGTTTAATGGCGAACAAAGCAAACCTGTCAAGGCCTTTGTGGAAGCAGTCATTCAGAGATTAAGGACTGCCGTAGACGATGATGTCTTTGTCCCTCTGTACCCCAAAAAGTTTTTAGAGGATAAGCGGTCACCGCAGTTCCAGTTTCAGAACAAACAGTTTTGGTCGGCGGTGAAGTTGCTAGGTAATATGGCATTATGGGGTGGTTTGATTCCTGAGCACATTCTTAAAGAGCTGATGTTGGAGAAACTTCTGGGCCGTTATTTGATGATAACCATTCTTAATGAATCTGATCCAAAACACACAGTCCAGAAATGCAAAAAGATTGCAGGTTGTTTTCCAGAATCATGGTTTATTGATTTAAATACTGGATCATCACTACCTCAGTTGCAAAACTTCAGCAAACATCTTCTTCAAACCGCACATCTGATATTCAAGGACAACAAGGACTCCAGAGGACTTTTATCTGATGTGCTGTTTGTTCTCAAGATCATCAAGGCTCATGACAGTATTAGGACTATCACTGAAAAATATAATTGCAAAGACCTGTTAAAGGCTCTCTAG</CDS>TAGCAGGCATCTTGTGTCAATGTGGTTTTATACAGTATGTTTTAAAAGTCACTTTGATATGTGTATGTATAGTGAATAAAATGTTTTATTTTGGAGCA</gene></seq><eos>"
seq4 = "RNA, {}, TLR1B, toll-like receptor 6 isoform X1<seq><gene>TTCCTTTGCTGAAGAAAGTTACTCGCAGTTGCCCAGGAGGCGTAACGACACATCCTATGAGTCGCTGAGGAGAGCTGCTGCCGGAGGTCACAAAACACACACAGTTTCTCAGAGGGTTGCTTCACACATAGGGGTAGAGCCGCGTGAGCGACGCCCTCACCAAGCAGATCTCTGAAGCTTCCCGTGGAGGTGGGTGACAGCAGAACGGACGCCTGTATCGTTAAGTCACACGCTGATCGCCAAGGAGTTTCAGTGAGACTTTACATATCGGGGGTAATCCTGAGGGTTCGCCTCTGCTTTTGATGGGATCTGTGGAAGAGTAAAGGTGTTGTCTTAGGAAGAATTCTAATGGATGAAGCTCGGCAGCCGGAGATGGTTTTTTCCCCATCGTGAACTGTAATTTGGACTATGAAAAAGTGATGTCCCAAATGGTTTTTGGAACAAAAAGTTGGTGAAGGAGCAATAGAGATGGAAACTCCCTCCTCCACCTTGGGTTTCACTGAAGTTTGTTATCACTCTCAGTTGAGTGCCGTTGCCTTCTAAAAGAGGCCTTGATAGGTTTCAGAGAAAAATCACAGAATCATAGAGATTGGAAGGGATCCCTGAAAAATACCTGCCTTTTAGTGAATATATAAGTTTAAAGAAAAGATGGAAGGAAACACCTTCGTTGTGTGACTTAATCTGACGACATTTATCAAGGCATAGAAGTATGCATAACTTTTGGGCAATGTATTCATCTTCTTCTAATGATTGCACAAGAGTGAGTGTCCTGGAATTGTGAGGAGCCTATGTAGTACTCTGAAGGGCAGCAGCAGTAATTCCATAAGATTAGAACAAAGACGGATGTATAAATATTGAGCATCTCCAAGGAACCATAAAGCAGT<CDS>ATGACAAAGAATATGAGATATCTCAGGAACTGTTTTATTTACAACTGTCTGTTTGTATTCACTTTCTGGGACAATATTGGCCTGGCTAAGAAAAATGAACTCTTTGCATCTGTTCCTAACAATTTTCTAGAAGATGGTTTGGACAAAAAAAATATGAGTTTTTCACACTCATACGCAAATAATCAGCACTACAAAGCTGATTATGGTTGGGTTGTGATAGAAAATACTATGGAAAGCCTATCATTGTCAGAAATCGCAGATGACAATGTAAGAAAATTAATAACTTTACTATCTAAATTTAGAAAAGGCTCCAGGCTACGAAATCTGACACTGACAAATATGTCAGTGGACTGGAAAGATATTATTAAAGTACTTCAGGTTGTATGGCACTCATCCATTGAATACTTCAATATCAACAACTTAACACAACTGGGGAACGTTGTAAGTACTCGCTTCGACTATTCAAAAACATCCATGAAAGCATTTGCAGTGAATAAGGTTTTAATCACAGATCTCTACTTTTCACAGGATGACGTTTATAACATATTTGCAAACATGAATATTGCAGCCTTGACAATAGCTGAATCTGAGTTAATTCATATGCTGTGTCCTTCATCTGACAGTCCCCTTAGATATATAAATTTCTCAAAGAATGATTTAACAGATCTGCTATTTCAGAACTGTGACAAACTAATTCAACTGGAGACATTTATTTTGCATAGGAATAAATTTGAAAGCCTTTCCAAGGTGAGCTTCATGACCAGCCGTATGAAATCACTGAGGTATCTGGACATGAGCAGCAACTTGCTGCGTAACAGTAGAGCTGAGGGGCGCTGTCAGTGGGCTGATTCTCTGGCAGAGCTGGACCTGTCCTCCAACCAGCTGACAGAGGCTGTGTTTGAGTGCTTGCCAGCTAATATCAACAAAGTGGACCTACAAAACAACCAGATTGCCAGTGTCCCCAAGGGGATCACTGAGCTGCACTCATTGCAGGAGCTGAACCTGGCATCCAACCGGCTGGCCGACCTGCCAGGGTGCAGGGCCTTTACAAGCCTGGAGATCCTGAACATAGAGAGGAATTTGATCCTCACCCCATCTGCCGACTTCTTTGAGACCTGCCCGAGCGTGAAGGAGCTGCAAGCTGGGCAAAACCCGTTCAAGTGTTCGTGTGAGCTGCAGGACTTCCTGCGCCTGGAGAGGCAGTCTGGGGGCAAGCTGTCCGGCTGGCCAGAGGCGTACGTGTGCAAGTACCCAGAAGACTTGAGCGGAACACAGCTGAAGGACTTCCACTTGACCGAGCTGGCTTGCAACACGACTCTGCTGCTTGTGACAGCTCTGCTGCTGACCCTGGTGCTGGTGGCTGTGGTGGCCTTTCTGTGCATCTACCTGGATGTGCCGTGGTACGTGCGGATGCTGTGGCAGTGGACGCAGACGAAGAGGAGAGCTTGGCACGACTGCCCCGAGGAGCGGGAAACTGCTCTGCAGTTCCACGCCTTCATTTCCTACAGCGAACGCGATTCCCTGTGGGTGAAGAATGAGCTGATCCCCAATTTGGAGAAGGGGGAGGGCTGCATCCAGCTGTGTCAGCATGAGAGGAACTTTATCCCTGGCAAAAGCATTGTGGAGAACATCATTAACTGCATTGAGAAGAGCTACAAGTCCATCTTTGTGTTGTCGCCCAACTTTGTGCAGAGCGAGTGGTGTCACTACGAGCTGTACTTTGCCCATCACAAGTTGTTTAGTGAGAACTCCAACAGCTTAATCCTGATCTTGCTGGAGCCAATCCCTCCATACGTTATCCCTGCGAGGTACCACAAGCTGAAGGCTCTCATGGCAAAGAGAACCTACCTGGAGTGGCCAAAGGAGAGGAGCAAGCATGCCCTTTTCTGGGCTAACCTGAGGGCAGCTATTAGCATCAATCTATCAGTGGCTGACGAACAGAACAGAACAGAAGTGTGA</CDS>TGAAAGATATTAAATGTCTTTATAATGCAGTTTCTTCTTCTCCTTGTTTGGTTTTTTTTTTATTTGTTTGTTTGTGAACCAATAAATACTTTATGATTTCCATTGATTTTCAATATA</gene></seq><eos>"
seq5 = "RNA, {}, GATAe, GATAe, isoform A<seq><gene>CATTGCGTCGCCGTCGTTGTTGTCATCGCAAGTGAGGTGCTTGGTGCTCAGAGTGCTCAGTGTTTAGTGTATCGAGCCCATAGACCGGAGACCAGTGCCACCACCAGAGCCCAGTTGGTCTCGAAAAAATAGCACTTCGTATCACTGCTAACTGGCAAAGGTTGCGCGCGGGCCACAACCACAATCGCCCCACGGGTGACAGCATTAGATCGCCCCCAAAGCTCCAAAGTTCCTACTTTCCCAAGTTACCAATTTTCGTCAATTTTCGCCAATTTTCCCCACTTGCTCACTTTGCGCGGCACTTTTTCGGGCGCCCAGATTGATAGGAGAAGTGAACCAAACACCTGGACAAGCCGTGCTTGTGATAAATTTTTGAAAACACAACAGAACTGAAATCAAGTGATAAAAACCAATCTTAGATAAAACGCTGGAAGAAACGCAAAGAAGCGGCGTAACTTCAATTTG<CDS>ATGGTCTGCAAAACTATCTCACCGAGTGTAAACATGCAGCTGAAGATGGAACAGCAGACAACGCAGCAGCAGCAGCAGCAACAGCAACAGCAACAGCAACAGCAACAATTGCAACAGCAACAGCATCAGGCTCTAACCAAGCAGCAGCTTCAACTGCTGGACAAAATCAAACTCGAGTCCAGCAACGGCGCCGATCAGTTGGCCCAGCAGACGGCCAACAATCTGGATGAGCAGCAGGAGCAGCAGCAACAACACCAACAGCAAGCCGCAACATCGGTGGGCGTGGTGGTGCAGACGGGCCAAGCGGGCGTCAGCGAACCGGAGGAGCAGTACGTGGTGGTGCCACGCAACCAGCGCCGTATTCTCACCACAGCCGGCACACTTGAATTAAACGAAGCTCGTGAAGGAGAACCGAGCACAAACGCCAGCAACGCGAGCTCAGGATCTGCTTCGGATAGCCACATCGAGTACCAGCGCAGTGCACACCAATCTCCCGGAGCCACTCACTACGTGCAGATGGCGCCACGCAACGCAGAGGTGACGGAACAGGTGGGAGCTGCTGCCGGAGCTCCGCCGGGCACCATCTTCGCCTATCCTATTATCTGCAACGGTGACGATGTGGCCGCGATCAAAATCGAAACACTCGAGAAGGGAGAGGCGACGGGGGAGTCGCAACAACAGCAGCAGCAACTGCAGCAACACCAGCATCAGCAGCAACAGCAATGTCCCACACCCAACGGAGCTAGCTACGGCGAGACCATAGTGATCAGCAGCGAGGCGGAGGCACTGCAGCATCACCACCAACAGCAGCAGCAGCACCAGCAGCAGCAGCATCAGCAGCAACACCACCAACACCAGGCGGCAGCTGCAGCATCGGCGGCGGCCCAGACGGTTCACATTGCCACCAGTTCGCATGGTGGAACCGTGCGCTTCGTTACGGAGGATGTACGCTTCACCACCGCCGGACCGGAGACATCCGCCTCGAATATGTACTACGATGTACCCGTGGTGGATGGCAGTGTGCATGCCAACGAGTCCAAGACCTACGCCGACCTAGGCAACGCATATGCCCCCTTCCCGCCCAGCTCCAGCTTCAGTAGCAATAGCTATGCGGCAACGCTTCAGCAGGGAAACACTATTTACAGTGTTCCGGGCACCGGTCAATTCTTGGCCAAGTCCGAGTCTGGCCTAAATCAGACTGGACTGCTGCGCCAGACTGGACCGGCTACTTTTCAGACGATATCTTTTGAGGGCGGCAACGGCATCGAGCCTTTATGGGCATCTCCCGCCCCGCCGGAGTACCAGAGTGTTCAATTTAGCAACTTCCATCCGCAGGTTATTGATGAATACGGGAGCGGCAATATGAGCACCTCCCACTGGCCGCCCGCCAGCAGCATAGGGCAGTACGATGGAAGTCTGGTGACCGCCTCGTCCACATCGTCGCCCAACCACGAGCTAAAGTGCGAGAACTGCCATGGGCCGTTCTTGCGGAAGGGTAGCGAGTACTTCTGTCCCAATTGCCCGGCCTTTATGCGGATGGCGCCCCGTATAACCCAGAGGCAGGCGAAGCCCAAGGCCGCCGCCGCCCCCAACAACCGGCGCAACGGGGTCACCTGCGCCAACTGTCAGACCAACTCGACGACCCTGTGGCGTCGCAACAACGAGGGCAATCCGGTGTGCAATGCCTGCGGACTGTACTACAAGCTGCACAACATGAACCGACCGCTGTCGATGAAGAAGGAGGGCATCCAGAAGCGCAAACGGAAGCCTAAGAACAACGGAGGAGCACCCATGCACCGAGCACCGCTGCCGAGCATGTCGCAGGGAGTCAACCTGATGGCCAACAGTCCGCTGTATCCTTCCCAAGTGCCAGTTAGCATGCTGAATAGCCAGCTTAATAGCCAGCAGAATTCCAGTCCGGAGCTGCACGACATGTCGACGACGGGTCAAGCGGGTGGGCAACGGGTGGTGTCCATTTCGCTGAATGCGACGGCGCCGCCCACCCCCGACGGAACGCTCAACATGTCCGCCCGTCATCACGTGACCGGCGAGAGCCATTCTCCTTACAGTCAGCAATCAACGCCACAGAGTCAATCGCCCCATCTGCCCGGCACGGTGCCGATAAACCGTCAGATAGTTCAGCCAGTACCCACCATCGAAAGCAGTCGCAGCTCCAACACTGAACTGACCCCCAGCGTAATCACGCGCACTGGTCTGCCAGAGCGATCATCGAATAACTAA</CDS>GCTAGCTTTGACTTATGGCTTAAGCTACGCACTAAAGTAACCTATAGTTAACCAAACCCAAATTTCAAAAATTGCCAAAATATTTATTTAAGCCCTAGCCGTTAGTCATACTATATAGCCAAGACCAATGTCCCAGTTAGAGGAAGATTTTGCCGGGTTTTCCGGGTCTTGAGTAACCATTTAACTGTGCAAATGTCTTCCTCCTTTTAGTCAACTGCTTACAGCAAATAGTTTTGAAATGATTTGTGCCAACCACTTAGGATCGCTATACTCTATATAGAGGTAGTTTAGCCCAAGACAGCACAAGCCCATAATGAAGCCTACGATTTTGTATGTTTAAGCCTCTTTTTAATTCTACAAAATAAATAAGTTGTAAACGC</gene></seq><eos>"
seq6 = "RNA, {}, str-143, Seven TM Receptor<seq><gene>ATAACTTC<CDS>ATGGAACTAACATTGAAATTGTGTCTTCATATAGCCCAATATGCAGGTTTCATAGTCGGGCAACTTACCAATTCGTGTCTTTTGTTTCTTATTTTTACTAGAGCTGAGAGATTATTTGGAAGTTATCGACATGTGATGGCTGTTTTTGCGTTGTTTTCTTTGGTTTATACTTGGATCGAGTTTATAGCACAGCCGGTGATGCACATAAAACAATCAATGTTCATTGTAATGCTAGATAGTCCTTTCACATTTGACGTTTCGACCGGGAATGAAATTACTTGTTTATACTGCTCATCATTTGCTCTTTGCATTTCACTTTTGGCTGCTCAGTTTTATTACCGATATATTGCTCTTTGCCAACCAGAAACCCTTGAGAAAATAAAAGGATGGAATCTGACCTTATTATTCATTCCTTGTATTGTTTGCTTTGTTGGTTGCGTTACTTGCGTCTATTTTGGAATGCACAATACTGTGGAAAAACAAAAATTTATGAGAGACGTGATGTTTGAGAACTATGATGTTGATTTGGGCAGGGAATCATTTATCGCAGCTTTTTATTGGTCATATGATAAAAATGGGTCTCGAATTTTCCGCTTTCGGGATACCATTGCGGCTTCTGGATGTATATTAGTGGCATGCTTCTCCACAATACTGTATTGTGCTTTCAAAATCTACATCAGACTCAAAAGTGCTCAAGCATTGATGAGTACTAAAACGAGAGAACTTAACCGACAGCTGTTTATCACTCTCACTTTTCAGACACTTTTTCCATTTTTCATGATGTATTGCCCTGTTGGTTCTTTGATTTCTTTTCCATTTCTTGAAATTGAAGTTGGGAAATTTGGTAATTACACTGGCGCAGCTGCCGGAATTTACCCAGCTTTAGAACCTCTTATTGCAATTTTCTGTATCAAAGATTTCAGAAGAGAAGTACTATGTCAACGGAAAAAAGTTCGAACTGAAGTAAAATCCAGATCAGGAATCCCATCTACTTCTTTCATATAA</CDS>CTTTTAAAAAAGTATATTTAACCTTTTTGAACTATGTATATTTAGCCTTTTTTTTCAAAGGTTTAGTTTCTTAATGCAATAAATTTTGAACCCTTA</gene></seq><eos>"
seq7 = "RNA, {}, CAT9, cationic amino acid transporter 9<seq><gene>CCGGGGGTGATTTCAGTCATCGTCTCGATCAATCTTCAACTCTCAAATTGAAGTTTCC<CDS>ATGGGAGGCCACGAAGGTTTCAGCAACCAACGCCTCTCCTCCGCCACATGGTTCTCTCATTTCCGTGCTTCAGCTCTCCGATCTAAATCACTGCCCCCTCCGTCCTCGCAGACTGCCGTTCGCTCCACCTCTGGCGACTCTCTGGTTCGTCGCCTCGGCCTTTTCGACCTCATACTGTTGGGCGTCGGCGCTTCCATCGGCGCCGGTGTCTTCGTCGTCACCGGAACCGTCGCTCGCGATGCTGGACCTGGAGTCACAATCAGCTTCTTGCTAGCTGGAGCATCGTGTGTGTTGAACGCTCTCTGTTATGCTGAACTTTCATCTCGTTTCCCTGCTGTTGTTGGTGGAGCTTACATGTACTCTTACTCAGCTTTCAATGAGATCACAGCTTTTCTTGTTTTCGTTCAACTCATGCTTGACTATCATATTGGCGCCGCTAGTATCTCCCGTAGCTTGGCCAGCTATGCTGTTGCTCTTCTTGAGCTTTTTCCAGCCCTCAAGGGCTCTATTCCTCTCTGGATGGGAAGCGGCAAAGAGCTCTTGGGAGGGCTTCTCTCTTTGAATATTTTGGCTCCCATTTTACTTGCTCTCTTGACCCTTGTTCTATGCCAAGGTGTGCGAGAATCTTCCGCTGTCAACTCCGTTATGACTGCTACTAAGGTCGTGATCGTTCTCGTTGTGATTTGTGCCGGTGCTTTTGAAATTGATGTAGCAAATTGGTCTCCTTTTGCGCCTAATGGTTTTAAAGCAGTTCTCACTGGAGCTACTGTTGTCTTCTTTTCATATGTTGGATTTGATGCAGTTGCTAATTCTGCTGAAGAATCGAAGAACCCTCAGCGCGATTTGCCAATCGGAATTATGGGGAGCCTTCTGGTTTGTATCTCGCTGTATATTGGCGTATGTTTGGTGCTCACCGGGATGGTCCCTTTTAGCCTCCTCAGCGAAGATGCTCCTTTAGCTGAAGCCTTTTCATCAAAGGGTATGAAGTTCGTCTCAATCTTAATCAGTATTGGTGCTGTTGCTGGACTCACAACTACTCTGCTTGTTGGTCTTTATGTTCAGTCCAGGTTATATCTGGGACTTGGAAGGGATGGTCTATTGCCTTCAATCTTTTCCAGAATACACCCAACTCTTCATACACCTCTTCATTCTCAAATCTGGTGCGGCATTGTAGCTGGCGTTTTAGCTGGCATATTTAATGTACATAGTCTTTCCCATATTCTGTCTGTTGGTACATTGACTGGGTATTCAGTTGTTGCGGCTTGTGTTGTGGCATTGAGACTGAATGATAAAAAAGATAGAGAGTCGTCCAATAGATGGACATCAAGTTGGCAGGAAGGCGTCATTTGCCTTGTTATAATTGCATGCAGTGGTTTTGGTGCTGGCGTATTTTACCGTTTCAGTGCTTCAGTTATATTTATTCTCCTCTCAGTTGGTGTGGCAGTTGTTGCTTCAGCAGTTCTTCATTATCGCCAAGCTTATGCTCTGCCTCTAGGTTCTGGGTTTAGCTGTCCTGGTGTACCCATTGTGCCATCCGTTTGCATTTTCTTCAACATCTTCTTATTTGCTCAGTTGCATTATGAAGCATGGATAAGATTTGTTGTTGTCAGTGTGCTTGCAACTGCCGTTTATGCCTTGTATGGGCAGTACCACGCAGATCCGAGCATGTTGGATTATCAGAGGGCACCTGAAACTGAAAGCGACGCTTAG</CDS>TAATGCTACATTATAGTTGTGTTACTTAAAGCCATTTGTAGAGTTGATAGCAAGTTTCTGCGTTTCGTGAGAGACATACTGTATGTCACATACATATTCAGGTCGAGTTTAAGGATTGGATGGGATAAGGCAGTTAAACATTATTGTTGTAAAGAAGTTAATAAAAGTATGCTTGTGCATTTCAACTAAAGCCTTGAATGACTGAAAATGGCTTTTGTGCTACCAAATAATCCCAGAAGCTAGCAATGTTCACAT</gene></seq><eos>"
seq8 = "RNA, {}, LOC4334864, ethylene-response factor C3<seq><gene>TTGTGCAAGTTGCATGGGGTTTGGGGAGGGGGAAGGAGCAGAGCAGTATATATAAAAGGGCAAAGCCTCCGTCCATTTATTCATACTCGATCTATTATTCGGCGGCGGCTGCCTCTGCTTGCTTGTGACTGTCGTGTCTGTGCGTGTGTGTGTACGTACATACTCCAGTATATAT<CDS>ATGCATTGCTGCATGTCGCTTCATCCTCACCGCCGCCACGGCGACGGCGACGTCGACGGATCAGCATCAGGATCAGGATCAGCGCGCCTCACCGCCGGCCTCATCAACTTCCTCGAATCGCGTCGCGCCGGCGCCATGAGCACCACCAACAGCTCATCCTCTGTCTCTGTCCCAGCCATGGACGCCCATGGACAGGAGGAGGAGGAGGAGCCGATGCAGGTGCAGCAACAGCAGGCGTTCCGCGGGGTGCGCAAGCGGCCATGGGGCAAGTTTGCGGCGGAGATCCGCGACTCGACGCGCAACGGCGTGCGCGTGTGGCTGGGCACGTTCGACAGCGCGGAGGAGGCGGCGCTGGCCTACGACCAGGCGGCGTTCGCCATGCGCGGGTCGGCGGCGGTGCTCAACTTCCCCATGGAGCAGGTGAGGCGTTCCATGGACATGTCCCTCCTGCAGGAAGGGGCGTCGCCGGTGGTGGCGCTGAAGCGGCGGCACTCCATGCGAGCGGCAGCAGCGGGGCGGCGGCGCAAGAGCGCTGCACCTGCACCGGCGGATCAGGAAGGCGGAGGAGGGGTGATGGAGCTGGAGGACCTGGGACCTGACTACCTGGAGGAGCTGCTAGCCGCCTCTCAGCCCATCGATATCACCTGCTGCACAAGCCCAAGCCACCACTCCATCTGA</CDS>TCTCTACTCGTATTCCTCACACGCGCAGAGGATGCACATCCTCTGCACTACACTTGTACTAGCAGTGGTTAACATGTGGATCCGGTTCCATGCTCTGATTGTTGCTTATATTTTTCAAGGAATATAAACAAATCCGATCCTGAGTTAATTAGTGTAAATATATAGATTTTGGGTTTCAGGATTACCATTTTTA</gene></seq><eos>"
seq9 = "RNA, {}, unknown gene, 3,5-epimerase/4-reductase<seq><gene>CCTAGTGAAATGCAGCCTGGCCGGGCGGGTGGGCTGTATACTCGAAGCACATACGCACCAAAGCGTGCACGCATAGCTCTCCTTCAACCAAGCCGGGGTTTTCCCTTACACAACTCACAACTCACAAAC<CDS>ATGGCGGGAGACAAGACCAACGGCGCTGCTGAGCCGGTGTTTCTGCTTTTCGGCAAGTCGGGCTGGATCGGTGGCCTGCTGCAGGAGGAGCTGAAGAAGCAGGGAGCCAAGTTCCACCTCGCGGACGCGCGTATGGAGGACCGCTCTGCCGTTGTTGCGGACATTGAGAAGTACAAGCCCACCCACGTGCTGAACGCGGCCGGCCTGACTGGCCGCCCCAACGTCGACTGGTGCGAGACGCACAAGCTGGAGACCATCCGCGCCAACGTGATTGGCTGCCTGACCCTGGCGGACGTGTGCAACCAGCGCGGCATCCACATGACCTACTACGGCACCGGCTGCATTTTCCACTACGACGATGACTTCCCGGTCAACAGCGGCAAGGGCTTCAAGGAGTCCGACAAGCCCAACTTCACCGGCTCCTACTACAGCCACACTAAGGCCATCGTGGAGGACCTGATCAAGCAGTACGACAACGTGCTGACCCTGCGCGTGCGCATGCCCATCATCGCCGACCTCACCTACCCTCGCAACTTCATCACCAAGATCATTAAGTACGACAAGGTCATCAACATCCCCAACAGCATGACTGTGCTGCCGGAGCTTCTGCCCATGTCCCTGGAGATGGCCAAGCGCGGCCTGACTGGCATCATGAACTTCACCAACCCCGGCGCCGTCAGCCACAACGAGATCCTTGAGATGTACAAGGAGTACATTGACCCCGAGTTCACCTGGTCCAACTTCTCGGTGGAGGAGCAGGCCAAGGTCATCGTGGCGCCCCGCTCCAACAACCTGCTGGACACGGCGCGGATAGAGGGTGAGTTCCCGGAGCTGCTGCCCATCAAGGAGTCGCTCCGGAAATACGTGTTCGAGCCCAACGCCGCTAAGAAGGATGAGGTGTACAAGGCGGTCAAGGAGATGCGCGGCCGGTAA</CDS>AGCGAAGCGGGGCCGTTCGTGGTCGGCGAAGTGGTTGGCGAGCTGACTGGTGCAGGGGTTTGAGGAGCGATCTGCGAGTGCGTGCGCAGGAGCAGCATGCAGCTGGATGAGAGTGCCTGTGAACGAGATGTGGAAGCGCCGGTCGTAGCTGGGGCTGAACATGGCTAGTGCTGACCCGTACGATACGGGGGCGGGGTGTGGTCGTGATGGGATGGCACGCACACGTGCTCTCTGTGCGGGGGCAGGTAGTGAACGCAGTGATTGGCCGCAAAGAAATGGGTGCAGGCTGTGGATGCAACGAGCATTAGAGCGTGGCGTGGCATGCGTGAAGGAGCGAGGCAGTGCAGGGCTTGCCCAGCAGGAAGGCAAGGCGGTAGCTTGCAAGGCGGAGAGTCACTTTTGAGCAGCTTTAAATGAAGTTTTGCTGACGCTACTAAGATCCCCATGGCGCGATATCTGATCGGCTATTGATTGCGTAAACTCGAAATGGCAATGTGCCGGTACAAAGGCTGTAGCGAATTGCTCAC</gene></seq><eos>"
seq10 = "RNA, {}, RAP1, DNA-binding transcription factor RAP1<seq><gene><CDS>ATGTCTAGTCCAGATGATTTTGAAACTGCACCAGCAGAATATGTTGATGCATTGGATCCGAGTATGGTCGTTGTTGACAGTGGTTCTGCTGCAGTCACAGCGCCATCGGATTCAGCCGCTGAGGTAAAAGCTAATCAAAACGAAGAAAACACTGGTGCTACTGCTGCTGAAACTAGTGAGAAAGTGGATCAGACGGAGGTCGAAAAGAAAGATGATGATGATACCACTGAAGTCGGTGTAACAACTACCACGCCTTCCATTGCTGATACTGCTGCTACCGCTAACATTGCTTCGACATCAGGTGCATCTGTTACTGAACCTACTACTGATGACACCGCTGCTGATGAGAAAAAGGAACAAGTAAGTGGTCCTCCTCTGTCAAATATGAAATTCTATCTTAATCGCGACGCGGATGCGCATGACTCTTTAAATGATATTGATCAATTAGCGCGTCTGATTAGAGCAAACGGTGGTGAAGTGCTAGACTCCAAGCCAAGAGAATCAAAAGAGAATGTTTTCATCGTGTCACCCTATAATCATACCAATTTACCAACGGTAACTCCGACTTATATTAAGGCATGTTGTCAAAGCAACTCGCTACTGAACATGGAAAATTATTTAGTACCATACGATAACTTTAGAGAAGTTGTCGATAGTAGGTTGCAGGAGGAATCGCATTCTAATGGTGTAGATAATAGCAATAGCAATAGCGATAACAAGGATTCTATCAGGCCCAAAACAGAAATAATCTCTACGAATACTAATGGCGCTACGGAAGACTCAACCAGCGAAAAAGTTATGGTAGACGCTGAACAACAAGCTAGATTACAAGAGCAAGCTCAGCTCCTTCGTCAACATGTAAGCAGCACCGCATCAATCACAAGCGGAGGGCACAATGACTTGGTCCAGATCGAACAACCTCAAAAAGATACCTCCAATAATAATAATAGCAACGTCAACGACGAAGACAATGATCTTTTGACACAGGACAACAACCCTCAAACAGCAGATGAGGGGAATGCATCTTTTCAAGCACAAAGGTCCATGATTTCGAGAGGCGCTTTGCCCTCCCACAATAAAGCTTCTTTTACAGATGAGGAAGATGAGTTTATTTTGGATGTTGTGAGAAAAAATCCAACCAGGCGTACAACACATACTCTTTACGATGAAATATCCCATTATGTGCCTAACCACACGGGTAATTCTATTAGGCACCGATTTAGAGTCTATCTTTCCAAAAGACTAGAGTACGTTTATGAGGTTGACAAGTTTGGTAAATTGGTTAGAGACGACGATGGAAATCTGATAAAGACTAAAGTTTTGCCACCATCAATTAAAAGGAAATTTTCAGCAGATGAAGATTACACTTTGGCAATCGCGGTCAAGAAGCAGTTTTATCGTGATTTGTTTCAAATCGACCCTGATACTGGAAGATCACTTATCACAGATGAGGATACACCCACTGCTATAGCGAGGAGGAATATGACAATGGATCCAAATCATGTGCCAGGGAGTGAACCCAACTTTGCAGCTTACAGAACCCAAAGCCGTAGGGGACCGATTGCACGAGAATTTTTCAAGCATTTTGCCGAAGAGCATGCAGCACATACTGAGAATGCTTGGAGAGATAGGTTTAGGAAGTTTCTTCTCGCTTACGGCATTGATGATTATATTAGTTATTATGAAGCTGAAAAAGCTCAGAATAGAGAGCCGGAACCGATGAAGAACCTCACCAATAGACCCAAGAGGCCTGGCGTTCCCACTCCTGGCAATTATAACTCTGCCGCCAAGAGGGCAAGAAATTATAGTTCTCAAAGAAATGTTCAGCCTACTGCCAATGCAGCGTCCGCTAATGCTGCTGCCGCTGCTGCTGCCGCTGCTTCCAACTCTTACGCTATACCAGAAAACGAATTACTGGACGAAGATACCATGAATTTCATTTCCAGTTTAAAGAATGATCTATCCAATATTTCAAATAGTTTGCCCTTTGAGTATCCACACGAGATTGCGGAAGCTATAAGAAGCGATTTTTCGAATGAAGATATATATGATAATATTGATCCTGATACCATAAGCTTTCCACCAAAAATTGCAACAACAGATCTTTTCCTGCCATTGTTCTTTCATTTTGGCAGTACGAGACAATTTATGGATAAACTTCATGAAGTTATTTCAGGAGATTATGAGCCATCACAGGCTGAAAAACTGGTACAGGATCTTTGCGATGAAACTGGTATACGTAAAAATTTCAGTACGAGCATATTGACGTGTTTATCCGGGGATCTGATGGTCTTTCCTCGCTATTTCTTGAACATGTTTAAGGACAATGTTAATCCTCCTCCCAACGTTCCTGGTATTTGGACACATGATGACGACGAATCGCTAAAAAGCAATGATCAAGAACAAATAAGGAAACTGGTTAAAAAGCATGGAACTGGTAGAATGGAAATGAGGAAAAGATTTTTTGAGAAGGACCTGTTATGA</CDS></gene></seq><eos>"

CROSS_SEQS = [seq1, seq2, seq4, seq3, seq5, seq6, seq7, seq8, seq9, seq10]

In [ ]:
PROJECT_ROOT = Path("..")
MODEL_DIR = PROJECT_ROOT / "model" / "GenNA"
TOKENIZER_PATH = PROJECT_ROOT / "configs" / "tokenizer.json"
TSV_PATH = PROJECT_ROOT / "data" / "species.tsv"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.bfloat16 if DEVICE.type == "cuda" else torch.float32
FLASH_ATTN_AVAILABLE = DEVICE.type == "cuda" and importlib.util.find_spec("flash_attn") is not None

TSNE_BATCH_SIZE = 64
TSNE_LAYER_INDEX = -18
TSNE_PERPLEXITY = 30
TSNE_LEARNING_RATE = 200
PPL_MAX_CTX = 4096

SINGLE_SPECIES = [
    "Boleophthalmus pectinirostris",
    "Periophthalmus magnuspinnatus",
    "Eucyclogobius newberryi",
    "Salmo salar",
    "Oncorhynchus mykiss",
    "Danio rerio",
    "Callorhinchus milii",
    "Rhincodon typus",
    "Falco naumanni",
    "Homo sapiens",
    "Drosophila melanogaster",
    "Saccharomyces cerevisiae",
    "Arabidopsis thaliana",
    "None",
]

CROSS_SPECIES = [
    "Homo sapiens",
    "Mus musculus",
    "Gallus gallus",
    "Danio rerio",
    "Drosophila melanogaster",
    "Caenorhabditis elegans",
    "Arabidopsis thaliana",
    "Oryza sativa Japanese Group",
    "Chlamydomonas reinhardtii",
    "Saccharomyces cerevisiae",
]

if not MODEL_DIR.exists():
    raise FileNotFoundError(f"Required path not found: {MODEL_DIR}")

if not TOKENIZER_PATH.exists():
    raise FileNotFoundError(f"Required path not found: {TOKENIZER_PATH}")

In [ ]:
sns.set_theme(style="white")
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
    "axes.unicode_minus": False,
    "axes.linewidth": 1.2,
    "axes.edgecolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
    "xtick.major.width": 1.2,
    "ytick.major.width": 1.2,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "xtick.major.size": 5,
    "ytick.major.size": 5,
})

In [ ]:
def build_tokenizer(tokenizer_path: Path) -> PreTrainedTokenizerFast:
    tokenizer_obj = Tokenizer.from_file(str(tokenizer_path))
    tokenizer = PreTrainedTokenizerFast(
        tokenizer_object=tokenizer_obj,
        unk_token="<unk>",
        pad_token="<pad>",
        eos_token="<eos>",
        model_max_length=4096,
    )
    tokenizer.padding_side = "left"
    tokenizer.truncation_side = "right"
    return tokenizer


def load_model(model_dir: Path, device: torch.device, dtype: torch.dtype):
    config = AutoConfig.from_pretrained(model_dir)
    model_kwargs = {
        "config": config,
        "torch_dtype": dtype,
    }
    if FLASH_ATTN_AVAILABLE:
        model_kwargs["attn_implementation"] = "flash_attention_2"
    model = AutoModelForCausalLM.from_pretrained(model_dir, **model_kwargs).to(device)
    model.eval()
    return model


def sequence_likelihood_no_bos_short(model, token_ids, device=None, max_ctx=None):
    model.eval()
    if device is None:
        device = next(model.parameters()).device

    token_ids = list(map(int, token_ids))
    n_tokens = len(token_ids)
    if n_tokens == 0:
        raise ValueError("token_ids is empty")

    if max_ctx is None:
        max_ctx = getattr(model.config, "n_positions", None) or getattr(model.config, "max_position_embeddings", None) or 4096
    if n_tokens > max_ctx:
        raise ValueError(f"Sequence length {n_tokens} exceeds model context {max_ctx}")

    input_ids = torch.tensor([token_ids], dtype=torch.long, device=device)
    with torch.no_grad():
        outputs = model(input_ids=input_ids)
        logits = outputs.logits.float()
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = input_ids[:, 1:].contiguous()
        log_probs = F.log_softmax(shift_logits, dim=-1)
        gathered = log_probs.gather(2, shift_labels.unsqueeze(-1)).squeeze(-1).squeeze(0)

    per_token_logprobs = [0.0] * n_tokens
    for i, lp in enumerate(gathered.tolist(), start=1):
        per_token_logprobs[i] = float(lp)

    total_logprob = float(sum(per_token_logprobs))
    n_predicted_tokens = max(1, n_tokens - 1)
    avg_neg_loglik = -total_logprob / n_predicted_tokens
    perplexity = float(np.exp(avg_neg_loglik))

    return {
        "total_logprob": total_logprob,
        "per_token_logprobs": per_token_logprobs,
        "n_tokens": n_tokens,
        "n_predicted_tokens": n_predicted_tokens,
        "avg_neg_loglik": avg_neg_loglik,
        "perplexity": perplexity,
    }


def fill_species_label(template: str, species: str) -> str:
    if species == "None":
        text = template.format("")
        text = text.replace("RNA, ,", "RNA,")
        text = text.replace("Genomic DNA, ,", "Genomic DNA,")
        return text
    return template.format(species)


def format_labels(labels, max_width=14):
    return [textwrap.fill(label, width=max_width, break_long_words=False) for label in labels]


def get_sequence_embeddings(texts, prefix="Genomic DNA, ", batch_size=16, layer_index=-18, pooling="mean"):
    all_embs = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc=f"Embedding {prefix.strip()}"):
            batch_texts = [prefix + t + "<eos>" for t in texts[i:i + batch_size]]
            enc = tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True)
            input_ids = enc["input_ids"].to(DEVICE)
            attention_mask = enc["attention_mask"].to(DEVICE)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
            hidden = outputs.hidden_states[layer_index]

            if pooling == "last":
                seq_lens = attention_mask.sum(dim=1) - 1
                batch_index = torch.arange(hidden.size(0), device=hidden.device)
                emb = hidden[batch_index, seq_lens]
            elif pooling == "mean":
                mask = attention_mask.unsqueeze(-1)
                summed = (hidden * mask).sum(dim=1)
                denom = mask.sum(dim=1).clamp(min=1)
                emb = summed / denom
            else:
                raise ValueError(f"Unknown pooling method: {pooling}")

            all_embs.append(emb.cpu())

    return torch.cat(all_embs, dim=0)

In [ ]:
tokenizer = build_tokenizer(TOKENIZER_PATH)
model = load_model(MODEL_DIR, DEVICE, DTYPE)
print(f"Model loaded on {DEVICE}")

In [ ]:
df_tax = pd.read_csv(TSV_PATH, sep="\t")
texts = df_tax["organism_name"].astype(str).tolist()
groups = df_tax["source_file"].astype(str).tolist()
df_tax.head()

In [ ]:
embeddings_dna = get_sequence_embeddings(
    texts,
    prefix="Genomic DNA, ",
    batch_size=TSNE_BATCH_SIZE,
    layer_index=TSNE_LAYER_INDEX,
    pooling="mean",
)

embeddings_rna = get_sequence_embeddings(
    texts,
    prefix="RNA, ",
    batch_size=TSNE_BATCH_SIZE,
    layer_index=TSNE_LAYER_INDEX,
    pooling="mean",
)

X = torch.cat([embeddings_dna, embeddings_rna], dim=0).float().numpy().astype(np.float32)
combined_groups = list(groups) + list(groups)
molecule_types = ["DNA"] * len(texts) + ["RNA"] * len(texts)

print("Running t-SNE...")
tsne = TSNE(n_components=2, perplexity=TSNE_PERPLEXITY, learning_rate=TSNE_LEARNING_RATE, random_state=42)
X_2d = tsne.fit_transform(X)

In [ ]:
mpl.rcParams.update({
    "font.size": 20,
    "axes.labelsize": 22,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 18,
    "legend.title_fontsize": 22,
    "axes.linewidth": 2,
    "xtick.major.width": 2,
    "ytick.major.width": 2,
    "xtick.major.size": 7,
    "ytick.major.size": 7,
})

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

ax2 = axes[0]
unique_types = ["DNA", "RNA"]
type_to_idx = {"DNA": 0, "RNA": 1}
colors_type = [type_to_idx[t] for t in molecule_types]
colors_list = ["#d62728", "#ff7f0e"]
cmap_type = mpl.colors.ListedColormap(colors_list)
ax2.scatter(X_2d[:, 0], X_2d[:, 1], c=colors_type, cmap=cmap_type, alpha=0.8, s=25, edgecolors="none")
handles2 = [
    plt.Line2D([], [], marker="o", linestyle="", label=t, markerfacecolor=colors_list[i], markersize=12, markeredgewidth=0)
    for i, t in enumerate(unique_types)
]
ax2.legend(handles=handles2, bbox_to_anchor=(1.15, 1.0), title="Molecule Type", loc="upper right", frameon=False)
ax2.set_xlabel("t-SNE 1")
ax2.set_ylabel("t-SNE 2")
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)
ax2.set_box_aspect(1)

ax1 = axes[1]
raw_groups = [g.replace(".txt", "") for g in combined_groups]
name_mapping = {
    "mammalian": "vertebrate mammalian",
    "non-mammalian": "vertebrate other",
}
clean_groups = [name_mapping.get(g, g) for g in raw_groups]
unique_groups = sorted(set(clean_groups))
group_to_idx = {g: i for i, g in enumerate(unique_groups)}
colors_group = [group_to_idx[g] for g in clean_groups]
cmap_group = plt.cm.tab20
ax1.scatter(X_2d[:, 0], X_2d[:, 1], c=colors_group, cmap=cmap_group, alpha=0.8, s=25, edgecolors="none")
handles1 = []
for i, g in enumerate(unique_groups):
    color = cmap_group(i / max(1, len(unique_groups) - 1)) if len(unique_groups) > 1 else cmap_group(0)
    handles1.append(plt.Line2D([], [], marker="o", linestyle="", label=g, markerfacecolor=color, markersize=12, markeredgewidth=0))
ax1.legend(handles=handles1, title="Species", bbox_to_anchor=(0.95, 1.0), loc="upper left", frameon=False)
ax1.set_xlabel("t-SNE 1")
ax1.set_ylabel("t-SNE 2")
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)
ax1.set_box_aspect(1.1)

for ax in [ax1, ax2]:
    ax.xaxis.set_major_locator(MultipleLocator(50))
    ax.yaxis.set_major_locator(MultipleLocator(50))

plt.tight_layout()
fig.subplots_adjust(wspace=0.45, right=0.95)
plt.show()

In [ ]:
results_single = []

if SINGLE_TEMPLATE:
    for spec in SINGLE_SPECIES:
        text = fill_species_label(SINGLE_TEMPLATE, spec)
        seq_tokens = tokenizer.encode(text)
        res = sequence_likelihood_no_bos_short(model, seq_tokens, device=DEVICE, max_ctx=PPL_MAX_CTX)
        results_single.append({
            "species": spec,
            "total_logprob": res["total_logprob"],
            "perplexity": res["perplexity"],
        })

    df_single = pd.DataFrame(results_single)
    df_single
    
else:
    df_single = pd.DataFrame()
    print("SINGLE_TEMPLATE is empty.")

In [ ]:
if not df_single.empty:
    plot_df_single = df_single.copy()
    plot_df_single["species"] = plot_df_single["species"].replace({"None": "Not specified"})
    plot_df_single = plot_df_single.sort_values("perplexity", ascending=False)

    species_list = plot_df_single["species"].tolist()
    ppl_values = plot_df_single["perplexity"].tolist()

    fig, ax = plt.subplots(figsize=(9, 7), dpi=300)
    highlight_color = "#D04040"
    normal_color = "#3B719F"
    control_color = "#808080"

    colors = []
    for sp in species_list:
        if "Boleophthalmus" in sp:
            colors.append(highlight_color)
        elif sp == "Not specified":
            colors.append(control_color)
        else:
            colors.append(normal_color)

    bars = ax.barh(species_list, ppl_values, color=colors, height=0.72, zorder=3, edgecolor="none")
    ax.set_yticks(range(len(species_list)))
    ax.set_yticklabels(species_list, fontsize=13)
    for label in ax.get_yticklabels():
        if label.get_text() != "Not specified":
            label.set_fontstyle("italic")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_xlabel("Perplexity", fontsize=16, labelpad=12)
    ax.tick_params(axis="x", labelsize=13)

    max_val = max(ppl_values)
    for bar, value, sp in zip(bars, ppl_values, species_list):
        text_color = highlight_color if "Boleophthalmus" in sp else "black"
        ax.text(value + (max_val * 0.02), bar.get_y() + bar.get_height() / 2, f"{value:.1f}", va="center", ha="left", fontsize=12, color=text_color)

    ax.set_xlim(0, max_val * 1.18)
    plt.tight_layout()
    plt.show()
else:
    print("df_single is empty.")

In [ ]:
results_cross = []

for seq, origin_spec in zip(CROSS_SEQS, CROSS_SPECIES):
    for anno in CROSS_SPECIES:
        seq_filled = seq.format(anno)

        seq_tokens = tokenizer.encode(seq_filled)
        res = sequence_likelihood_no_bos_short(
            model,
            seq_tokens,
            device=DEVICE,
            max_ctx=PPL_MAX_CTX,
        )

        results_cross.append({
            "origin": origin_spec,
            "replace": anno,
            "total_logprob": res["total_logprob"],
            "perplexity": res["perplexity"],
        })

df_cross = pd.DataFrame(results_cross)
df_cross.head()

In [ ]:
mat_perplex = df_cross.pivot(
    index="origin",
    columns="replace",
    values="perplexity",
)

mat_perplex = mat_perplex.reindex(
    index=CROSS_SPECIES,
    columns=CROSS_SPECIES,
)

baseline_ppls = np.diag(mat_perplex)
mat_delta = mat_perplex.sub(baseline_ppls, axis=0)

mat_delta

In [ ]:
if not mat_delta.empty:
    plt.figure(figsize=(7, 6), dpi=300)
    ax = sns.heatmap(
        mat_delta,
        cmap="Reds",
        vmin=0,
        vmax=100,
        linewidths=0.5,
        linecolor="white",
        square=True,
        annot=True,
        fmt=".0f",
        annot_kws={"size": 9},
        cbar_kws={"shrink": 0.8},
    )

    ax.set_xticklabels(mat_delta.columns, rotation=90, fontsize=10, fontstyle="italic")
    ax.set_yticklabels(mat_delta.index, rotation=0, fontsize=10, fontstyle="italic")
    ax.set_xlabel("Provided Species Label (Prompt)", fontsize=12, labelpad=10)
    ax.set_ylabel("True Species Label", fontsize=12, labelpad=10)

    cbar = ax.collections[0].colorbar
    cbar.ax.tick_params(labelsize=10)
    cbar.set_label("$\\Delta$ Perplexity (Penalty)", fontsize=12)
    cbar.outline.set_visible(False)

    plt.tight_layout()
    plt.show()
else:
    print("mat_delta is empty.")

In [ ]:
DESIRED_ORDER_LEFT = [
    "Oryza sativa Japanese Group",
    "Arabidopsis thaliana",
    "Chlamydomonas reinhardtii",
    "Saccharomyces cerevisiae",
    "Caenorhabditis elegans",
    "Drosophila melanogaster",
    "Danio rerio",
    "Gallus gallus",
    "Mus musculus",
    "Homo sapiens",
]

REF_SPECIES_ORDER = [
    "Homo sapiens",
    "Mus musculus",
    "Gallus gallus",
    "Danio rerio",
    "Drosophila melanogaster",
    "Caenorhabditis elegans",
    "Saccharomyces cerevisiae",
    "Arabidopsis thaliana",
    "Oryza sativa Japanese Group",
    "Chlamydomonas reinhardtii",
]

REF_LINKAGE = np.array([
    [0, 1, 1.0, 2],
    [10, 2, 2.0, 3],
    [11, 3, 3.5, 4],
    [4, 5, 2.5, 2],
    [12, 13, 6.0, 6],
    [14, 6, 9.0, 7],
    [7, 8, 1.5, 2],
    [16, 9, 4.0, 3],
    [15, 17, 15.0, 10],
])

available_species = [s for s in DESIRED_ORDER_LEFT if s in mat_delta.index and s in mat_delta.columns]

if len(available_species) < 2:
    print("Not enough species available in mat_delta for clustering.")
else:
    real_delta = mat_delta.loc[available_species, available_species].copy()

    sym_distance = (real_delta.values + real_delta.values.T) / 2
    np.fill_diagonal(sym_distance, 0)

    condensed_dist = squareform(sym_distance)
    linkage_matrix_llm = sch.linkage(
        condensed_dist,
        method="ward",
        optimal_ordering=True,
    )

    llm_labels_wrapped = format_labels(available_species, max_width=12)
    ref_labels_wrapped = format_labels(REF_SPECIES_ORDER, max_width=12)

    fig, axes = plt.subplots(1, 2, figsize=(8, 8), dpi=300)

    ax0 = axes[0]
    sch.dendrogram(
        linkage_matrix_llm,
        labels=llm_labels_wrapped,
        orientation="left",
        leaf_font_size=10,
        color_threshold=0.7 * max(linkage_matrix_llm[:, 2]),
        above_threshold_color="gray",
        count_sort="ascending",
        ax=ax0,
    )
    ax0.spines["top"].set_visible(False)
    ax0.spines["right"].set_visible(False)
    ax0.spines["left"].set_visible(False)
    ax0.set_xlabel("Genomic Distance\n(PPL Penalty)", fontsize=10, fontweight="bold")
    ax0.set_title("a. LLM-Based Tree\n", fontsize=11, fontweight="bold", loc="left", pad=10)

    for label in ax0.get_yticklabels():
        label.set_fontstyle("italic")
        label.set_verticalalignment("center")

    ax1 = axes[1]
    sch.dendrogram(
        REF_LINKAGE,
        labels=ref_labels_wrapped,
        orientation="left",
        leaf_font_size=10,
        color_threshold=8,
        above_threshold_color="gray",
        ax=ax1,
    )
    ax1.spines["top"].set_visible(False)
    ax1.spines["right"].set_visible(False)
    ax1.spines["left"].set_visible(False)
    ax1.spines["bottom"].set_visible(False)

    ax1.set_xlabel("")
    ax1.set_xticks([])
    ax1.tick_params(axis="x", bottom=False, labelbottom=False)
    ax1.set_title("b. Reference Tree\n", fontsize=11, fontweight="bold", loc="left", pad=10)

    for label in ax1.get_yticklabels():
        label.set_fontstyle("italic")
        label.set_verticalalignment("center")

    plt.tight_layout()
    plt.show()